# 🧠 Lab 01: ทำความเข้าใจ Concept k-NN ด้วย Pandas & Matplotlib
**Course:** 1322308 Machine Learning  
**Core Objective:** เข้าใจกลไกและคณิตศาสตร์ของ **k-Nearest Neighbors (k-NN)** แบบเห็นภาพจริงทีละสเต็ป โดยใช้เพียง **Pandas** และ **Matplotlib** เท่านั้น (ไม่พึ่งพา Scikit-Learn หรือโมเดลกล่องดำ)

---

### 📌 หัวใจสำคัญของ k-NN (The Core Concept)
อัลกอริทึม k-NN ไม่มีสมการเส้นตรง ไม่มีการปรับจูนค่าน้ำหนัก (Weight/Bias) ล่วงหน้า จัดเป็น **Lazy Learner (Instance-based Learning)** ซึ่งทำงานเพียง 4 ขั้นตอนพื้นฐาน:
1. **คำนวณระยะห่าง (Distance):** วัดว่าจุดข้อมูลใหม่ที่เราไม่รู้คลาส อยู่ห่างจากจุดข้อมูลในอดีตแต่ละจุดเท่าไร
2. **เรียงลำดับ (Sorting):** เรียงระยะห่างจากน้อยที่สุดไปหามากที่สุด
3. **คัดเลือกเพื่อนบ้าน (Select Neighbors):** เลือกจุดที่อยู่ใกล้ที่สุดจำนวน $k$ จุดแรก
4. **ออกเสียงข้างมาก (Majority Vote):** ดูว่าเพื่อนบ้าน $k$ จุดนั้น ส่วนใหญ่เป็นคลาสอะไร จุดใหม่ก็จะเป็นคลาสนั้น


## 1. นำเข้าไลบรารีพื้นฐาน (Imports)
เราจะใช้เพียง 2 ไลบรารีหลัก:
- `pandas`: สำหรับเก็บข้อมูลในรูปตาราง (DataFrame) และคำนวณระยะทางแบบ Vectorized Operations
- `matplotlib.pyplot`: สำหรับพล็อตจุดข้อมูล จุดทดสอบ และวาดวงกลมรัศมีเพื่อนบ้านเพื่อดูภาพจำลองจริง


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ตรวจสอบและตั้งค่า Font ภาษาไทยสำหรับ Matplotlib (แก้ปัญหาสระ/วรรณยุกต์เพี้ยนและตัวอักษรสี่เหลี่ยม)
thai_fonts = ['Leelawadee UI', 'Tahoma', 'Angsana New', 'Cordia New', 'TH Sarabun New']
available_fonts = [f.name for f in fm.fontManager.ttflist]
selected_font = next((f for f in thai_fonts if f in available_fonts), 'sans-serif')

plt.rcParams['font.family'] = selected_font
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (9, 6)
plt.rcParams['axes.unicode_minus'] = False  # ป้องกันเครื่องหมายลบ (-) แสดงผลเป็นสี่เหลี่ยม

print(f"✅ โหลดสำเร็จ: ตั้งค่า Font ภาษาไทยเป็น '{selected_font}' ใน Matplotlib เรียบร้อย")


### 💡 คำอธิบาย
การใช้เพียง Pandas ช่วยให้เราเห็นว่า **k-NN แท้จริงแล้วเป็นเพียงการ Manipulate ตารางข้อมูล** (สร้างคอลัมน์คำนวณระยะห่าง $\rightarrow$ จัดเรียงแถว $\rightarrow$ ดึงหัวตาราง $\rightarrow$ นับความถี่) โดยไม่ต้องมีโค้ด Machine Learning ซับซ้อนเลย


---
## 2. สร้างชุดข้อมูลตัวอย่างที่เข้าใจง่าย (Toy Dataset)
สมมติโจทย์ **"การจำแนกผลไม้: ส้ม (Orange) vs มะนาว (Lemon)"** โดยมี 2 คุณลักษณะ (Features):
1. **น้ำหนัก (Weight)**: มีหน่วยเป็นกรัม (g)
2. **ความหวาน (Sweetness)**: มีระดับคะแนน 1 ถึง 10

เราสร้างข้อมูลตัวอย่างของผลไม้ที่มีการติดป้ายชื่อ (Label) ไว้แล้ว 10 ลูก:


In [ ]:
# ข้อมูลผลไม้ที่มีการติดป้ายชื่อไว้แล้ว (Training Data)
data = {
    'Fruit_ID': [f'F{i+1:02d}' for i in range(10)],
    'Weight':    [120, 130, 125, 140, 135, 180, 190, 185, 175, 195],
    'Sweetness': [  3,   2,   4,   3,   4,   8,   9,   7,   8,   9],
    'Class':     ['Lemon', 'Lemon', 'Lemon', 'Lemon', 'Lemon', 
                  'Orange', 'Orange', 'Orange', 'Orange', 'Orange']
}

df = pd.DataFrame(data)
print("=== ตารางข้อมูลผลไม้ 10 ตัวอย่างแรก ===")
display(df)


### 💡 สำรวจข้อมูลเชิงตรรกะ
- **Lemon (มะนาว):** น้ำหนักเบากว่า (120 - 140g) และหวานน้อยกว่า (ความหวาน 2 - 4)
- **Orange (ส้ม):** น้ำหนักมากกว่า (175 - 195g) และหวานมากกว่า (ความหวาน 7 - 9)
สังเกตว่าผลไม้ทั้งสองชนิดเกิดการเกาะกลุ่มกันตามธรรมชาติในพื้นที่มิติข้อมูล


---
## 3. วาดภาพการกระจายตัวของข้อมูลด้วย Matplotlib (Visualizing Data)
ก่อนจะทำนายผล เรามาพล็อตตำแหน่งของ Lemon และ Orange บนแกน $X$ (Weight) และแกน $Y$ (Sweetness)


In [ ]:
plt.figure(figsize=(9, 6))

# กรองแยกกลุ่มตามคลาสผลไม้
lemons = df[df['Class'] == 'Lemon']
oranges = df[df['Class'] == 'Orange']

# พล็อตจุด Lemon (สีเขียวมะนาว) และ Orange (สีส้ม)
plt.scatter(lemons['Weight'], lemons['Sweetness'], color='lime', edgecolor='black', s=120, label='Lemon (มะนาว)')
plt.scatter(oranges['Weight'], oranges['Sweetness'], color='darkorange', edgecolor='black', s=120, label='Orange (ส้ม)')

# ใส่ป้ายกำกับ ID ของผลไม้แต่ละลูก
for _, row in df.iterrows():
    plt.annotate(f"{row['Fruit_ID']}", (row['Weight'] + 1.2, row['Sweetness'] + 0.1), fontsize=9)

plt.title("การกระจายตัวของข้อมูลผลไม้ (Fruit Dataset in 2D Space)", fontsize=14, fontweight='bold')
plt.xlabel("Weight (น้ำหนัก: กรัม)")
plt.ylabel("Sweetness (ระดับความหวาน: 1-10)")
plt.xlim(110, 210)
plt.ylim(0, 11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()


### 💡 สิ่งที่สังเกตได้จากกราฟ
เราเห็นจุดสีเขียว (Lemon) กระจุกตัวอยู่มุมล่างซ้าย และจุดสีส้ม (Orange) กระจุกตัวอยู่มุมบนขวาอย่างชัดเจน


---
## 4. มีผลไม้ปริศนาลูกใหม่เข้ามา (New Query Point)
สมมติว่าเราเก็บผลไม้ปริศนาลูกหนึ่งมาได้:
- **น้ำหนัก = 150 กรัม**
- **ความหวาน = 6.0**
- เราต้องการทายว่า: **"ผลไม้ลูกนี้คือ Lemon หรือ Orange?"**


In [ ]:
new_fruit = {'Weight': 150, 'Sweetness': 6.0}
print(f"🍏 ผลไม้ปริศนา (Unknown Point): น้ำหนัก = {new_fruit['Weight']}g, ความหวาน = {new_fruit['Sweetness']}")

# วาดกราฟเปรียบเทียบตำแหน่งของผลไม้ลูกใหม่บนแผนผังเดิม
plt.figure(figsize=(9, 6))
plt.scatter(lemons['Weight'], lemons['Sweetness'], color='lime', edgecolor='black', s=120, label='Lemon')
plt.scatter(oranges['Weight'], oranges['Sweetness'], color='darkorange', edgecolor='black', s=120, label='Orange')

# พล็อตจุดผลไม้ปริศนาเป็นรูปดาวสีแดง ⭐
plt.scatter(new_fruit['Weight'], new_fruit['Sweetness'], color='red', marker='*', s=350, edgecolor='black', label='Unknown Fruit (?)')
plt.annotate("Unknown (?)", (new_fruit['Weight'] + 2, new_fruit['Sweetness']), fontsize=12, fontweight='bold', color='red')

plt.title("ตำแหน่งของผลไม้ปริศนาเทียบกับข้อมูลเดิม", fontsize=14, fontweight='bold')
plt.xlabel("Weight (กรัม)")
plt.ylabel("Sweetness (1-10)")
plt.xlim(110, 210)
plt.ylim(0, 11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()


### 💡 คำถามเชิงสัญชาตญาณ
จากสายตามนุษย์ เรามองเห็นว่าจุดสีแดงอยู่กึ่งกลางระหว่างสองกลุ่ม แต่จุดนี้ใกล้กับผลไม้ลูกไหนมากที่สุดกันแน่?  
**นี่คือหน้าที่ของ k-NN ที่จะวัดระยะทางเชิงตัวเลขออกมาให้ชัดเจน!**


---
## 5. ขั้นตอนที่ 1: คำนวณระยะทางด้วย Pandas (Distance Calculation)
สูตรระยะทางที่นิยมที่สุดคือ **Euclidean Distance ($L_2$)**:
$$\text{Distance} = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$

ใน Pandas เราสามารถนำคอลัมน์มาลบกันและยกกำลังได้โดยตรงแบบ Vectorized Operation:


In [ ]:
# คำนวณระยะทาง Euclidean ระหว่างผลไม้แต่ละลูกกับจุดใหม่
# ((Weight - New_Weight)^2 + (Sweetness - New_Sweetness)^2) ^ 0.5
df['Distance'] = (
    (df['Weight'] - new_fruit['Weight'])**2 + 
    (df['Sweetness'] - new_fruit['Sweetness'])**2
)**0.5

# ปัดทศนิยม 2 ตำแหน่งให้อ่านง่าย
df['Distance'] = df['Distance'].round(2)

print("=== ตารางข้อมูลพร้อมคอลัมน์ระยะทาง (Distance) ===")
display(df)


### 💡 วิเคราะห์สิ่งที่เกิดขึ้นใน Pandas
คอลัมน์ `Distance` บอกว่าผลไม้ลูกปริศนา อยู่ห่างจาก `F01` เท่ากับ 30.15 หน่วย, อยู่ห่างจาก `F05` เท่ากับ 15.13 หน่วย ฯลฯ


---
## 6. ขั้นตอนที่ 2 & 3: เรียงลำดับและคัดเลือก $k$ เพื่อนบ้าน (Sort & Select Neighbors)
นำตารางมาเรียงลำดับตามคอลัมน์ `Distance` จากน้อยไปมาก (`sort_values`) แล้วเลือก $k$ แถวแรก (`head(k)`)


In [ ]:
# กำหนดค่า k เช่น k = 3
k = 3

# เรียงลำดับจากใกล้สุดไปไกลสุด
df_sorted = df.sort_values(by='Distance').reset_index(drop=True)

# เลือกเพื่อนบ้านที่ใกล้ที่สุด k อันดับแรก
k_neighbors = df_sorted.head(k)

print(f"=== เพื่อนบ้านที่ใกล้ที่สุด {k} อันดับแรก (k = {k}) ===")
display(k_neighbors[['Fruit_ID', 'Weight', 'Sweetness', 'Class', 'Distance']])


### 💡 สิ่งที่ตารางบอกเรา
เพื่อนบ้าน 3 ลูกที่ใกล้ที่สุดคือ:
1. `F05` (Lemon) - ห่างเพียง 15.13
2. `F04` (Lemon) - ห่างเพียง 10.44  *(ใกล้ที่สุด!)*
3. `F09` (Orange) - ห่าง 25.08


---
## 7. ขั้นตอนที่ 4: ออกเสียงข้างมาก (Majority Voting)
นับจำนวนคลาสของเพื่อนบ้านทั้ง $k$ ตัว คลาสที่มีจำนวนเสียงมากที่สุด (`mode()` หรือ `value_counts()`) จะเป็นคำตอบ:


In [ ]:
# นับคะแนนเสียงจากเพื่อนบ้าน k ตัว
votes = k_neighbors['Class'].value_counts()
print(f"ผลการนับคะแนนเสียงโหวต (สำหรับ k = {k}):")
for cls, count in votes.items():
    print(f"- {cls}: {count} เสียง")

# คลาสที่ได้คะแนนสูงสุด
predicted_class = k_neighbors['Class'].mode()[0]
print(f"\n🎯 สรุปคำทำนาย: ผลไม้ลูกนี้คือ >>> {predicted_class} <<<")


### 💡 วิเคราะห์ผลลัพธ์
ในบรรดาเพื่อนบ้าน 3 ตัว มี **Lemon 2 เสียง** และ **Orange 1 เสียง**  
ดังนั้นผลการโหวตข้างมากจึงตัดสินว่าผลไม้ลูกนี้คือ **Lemon**!


---
## 8. วาดภาพวงรัศมีของเพื่อนบ้านด้วย Matplotlib (Visualizing the k-Neighbors)
เพื่อให้เห็นภาพชัดเจนที่สุด เราจะใช้ Matplotlib วาด:
1. **วงกลมรัศมี (Neighborhood Boundary):** ขนาดรัศมีเท่ากับระยะทางของเพื่อนบ้านตัวที่ $k$
2. **เส้นเชื่อมโยง (Connecting Lines):** ลากเส้นประจากผลไม้ปริศนาไปยังเพื่อนบ้าน $k$ ตัวที่โหวต


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# 1. พล็อตข้อมูลทั้งหมด
ax.scatter(lemons['Weight'], lemons['Sweetness'], color='lime', edgecolor='black', s=130, label='Lemon', zorder=3)
ax.scatter(oranges['Weight'], oranges['Sweetness'], color='darkorange', edgecolor='black', s=130, label='Orange', zorder=3)

# 2. พล็อตจุดผลไม้ปริศนา
ax.scatter(new_fruit['Weight'], new_fruit['Sweetness'], color='red', marker='*', s=400, edgecolor='black', label='Unknown Fruit', zorder=5)

# 3. ลากเส้นประไปยังเพื่อนบ้าน k ตัว
for _, neighbor in k_neighbors.iterrows():
    ax.plot(
        [new_fruit['Weight'], neighbor['Weight']],
        [new_fruit['Sweetness'], neighbor['Sweetness']],
        color='purple', linestyle=':', linewidth=2, zorder=2
    )

# 4. วาดวงกลมรัศมีครอบคลุมเพื่อนบ้าน k ตัว
max_radius = k_neighbors['Distance'].max()
circle = plt.Circle(
    (new_fruit['Weight'], new_fruit['Sweetness']),
    max_radius,
    color='purple', fill=True, alpha=0.12, linestyle='--', linewidth=2, zorder=1
)
ax.add_patch(circle)

# ป้ายอธิบาย ID
for _, row in df.iterrows():
    ax.annotate(f"{row['Fruit_ID']}", (row['Weight'] + 1.2, row['Sweetness'] + 0.1), fontsize=9)

ax.set_title(f"ภาพจำลองขอบเขตรัศมีการโหวตของ k-NN (k = {k}, Radius = {max_radius:.2f})", fontsize=14, fontweight='bold')
ax.set_xlabel("Weight (กรัม)")
ax.set_ylabel("Sweetness (1-10)")
ax.set_xlim(110, 210)
ax.set_ylim(0, 11)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper left')
plt.show()


### 💡 ความหมายของวงกลมสีม่วง
- **วงกลมสีม่วง** คือขอบเขตที่กวาดไปเจอเพื่อนบ้านครบ $k$ ตัวพอดี
- จุดใดที่อยู่นอกวงกลมสีม่วง จะ**ไม่มีสิทธิ์ออกเสียงโหวต**เลย
- จุดที่อยู่ในวงกลม มี Lemon 2 จุด และ Orange 1 จุด ชี้ให้เห็นอย่างโปร่งใสว่าทำไมโมเดลถึงทายว่าเป็น Lemon


---
## 9. ผลกระทบของการเปลี่ยนค่า $k$ (The Impact of Changing $k$)
จะเกิดอะไรขึ้นถ้าเราเปลี่ยนค่า $k$ เป็น 1, 3, 5, และ 7?  
มาเขียนลูปง่ายๆ ด้วย Pandas ทดสอบดู:


In [ ]:
k_values = [1, 3, 5, 7]

print(f"{'k':<4} | {'คะแนนโหวต (Votes)':<25} | {'ผลลัพธ์ (Prediction)':<15}")
print("-" * 50)

for k_test in k_values:
    neighbors_test = df_sorted.head(k_test)
    vote_counts = dict(neighbors_test['Class'].value_counts())
    pred = neighbors_test['Class'].mode()[0]
    
    votes_str = ", ".join([f"{cls}: {cnt}" for cls, cnt in vote_counts.items()])
    print(f"{k_test:<4} | {votes_str:<25} | {pred:<15}")


### 💡 บทเรียนเรื่องการเลือกค่า $k$:
1. **เมื่อ $k = 1$:** โมเดลจะดูแค่จุดที่ใกล้ที่สุดจุดเดียว (`F04` ซึ่งเป็น Lemon) ทำให้ไวต่อสัญญาณรบกวน (Noise/Outlier) สูง เสี่ยงต่อ **Overfitting**
2. **ทำไมจึงควรเลือก $k$ เป็นเลขคี่ (Odd Number)?**  
   ในปัญหา 2 คลาส หากเลือก $k=2$ หรือ $k=4$ อาจเกิดสถานการณ์ที่คะแนนโหวตเท่ากัน (เช่น Lemon 2 เสียง, Orange 2 เสียง) ทำให้โมเดลตัดสินไม่ได้ว่าคลาสใดชนะ
3. **เมื่อ $k$ มีค่ามากเกินไป (เช่น $k=9$ หรือ $k=N$):**  
   วงกลมจะขยายใหญ่จนครอบคลุมข้อมูลเกือบทั้งหมด ผลการทำนายจะถูกกลืนด้วยคลาสที่มีจำนวนเยอะที่สุดในตาราง เกิดภาวะ **Underfitting**


---
## 10. ความลับที่สำคัญที่สุด: ทำไมต้องทำ Feature Scaling?
สังเกตตัวเลขในชุดข้อมูลของเรา:
- `Weight`: มีค่าระดับ **100 - 200** (ผลต่างอาจเป็น 10 - 50)
- `Sweetness`: มีค่าระดับ **1 - 10** (ผลต่างอย่างมากคือ 1 - 5)

เมื่อนำมาคำนวณ $(x_1 - x_2)^2$:
- ผลต่างของ Weight: $20^2 = 400$
- ผลต่างของ Sweetness: $2^2 = 4$

**ฟีเจอร์ Weight มีอิทธิพลต่อระยะทางมากกว่า Sweetness ถึง 100 เท่า!** ทำให้ Sweetness แทบไม่มีความหมาย  
เพื่อแก้ปัญหานี้ เราสามารถทำ **Min-Max Normalization** ให้อยู่ในช่วง 0 ถึง 1 ด้วย Pandas ได้ง่ายๆ:
$$\text{Value}_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$


In [ ]:
# ทำ Min-Max Normalization ด้วยคำสั่งพื้นฐานของ Pandas
df_scaled = df.copy()

weight_min = df['Weight'].min()
weight_max = df['Weight'].max()
sweet_min  = df['Sweetness'].min()
sweet_max  = df['Sweetness'].max()

# สเกลข้อมูลในตาราง
df_scaled['Weight_norm'] = (df['Weight'] - weight_min) / (weight_max - weight_min)
df_scaled['Sweetness_norm'] = (df['Sweetness'] - sweet_min) / (sweet_max - sweet_min)

# สเกลจุดผลไม้ปริศนาด้วย Min/Max เดียวกัน
new_fruit_norm = {
    'Weight_norm': (new_fruit['Weight'] - weight_min) / (weight_max - weight_min),
    'Sweetness_norm': (new_fruit['Sweetness'] - sweet_min) / (sweet_max - sweet_min)
}

# คำนวณระยะทางใหม่บนฟีเจอร์ที่ Normalize แล้ว
df_scaled['Distance_norm'] = (
    (df_scaled['Weight_norm'] - new_fruit_norm['Weight_norm'])**2 + 
    (df_scaled['Sweetness_norm'] - new_fruit_norm['Sweetness_norm'])**2
)**0.5
df_scaled['Distance_norm'] = df_scaled['Distance_norm'].round(3)

print("=== ตารางข้อมูลหลังทำ Feature Scaling (ค่าอยู่ระหว่าง 0 ถึง 1) ===")
display(df_scaled[['Fruit_ID', 'Class', 'Weight_norm', 'Sweetness_norm', 'Distance_norm']].sort_values(by='Distance_norm').head(5))


### 💡 บทเรียนเรื่อง Feature Scaling
เมื่อสเกลข้อมูลให้อยู่ในช่วง $[0, 1]$ เท่ากันแล้ว ทั้งสองฟีเจอร์จะมีน้ำหนักต่อการตัดสินใจเท่าเทียมกันอย่างแท้จริง  
**ดังนั้น กฎเหล็กของ k-NN คือ: ต้อง Normalize หรือ Standardize ข้อมูลเสมอก่อนนำไปใช้!**


---
## 11. เปรียบเทียบระยะทาง: Euclidean ($L_2$) vs Manhattan ($L_1$)
นอกจากเส้นตรง (Euclidean) แล้ว เรายังวัดระยะทางแบบตามแนวแกนตารางหมากรุก (Manhattan) ได้:
$$\text{Manhattan Distance} = |x_1 - x_2| + |y_1 - y_2|$$


In [ ]:
# คำนวณระยะ Manhattan ด้วย Pandas
df_scaled['Manhattan_Dist'] = (
    (df_scaled['Weight_norm'] - new_fruit_norm['Weight_norm']).abs() + 
    (df_scaled['Sweetness_norm'] - new_fruit_norm['Sweetness_norm']).abs()
).round(3)

comparison = df_scaled[['Fruit_ID', 'Class', 'Distance_norm', 'Manhattan_Dist']].sort_values(by='Distance_norm').head(5)
print("=== เปรียบเทียบระยะ Euclidean vs Manhattan สำหรับ 5 เพื่อนบ้านแรก ===")
display(comparison)


### 💡 ข้อแตกต่าง
- **Euclidean ($L_2$):** วัดระยะเส้นทแยงมุม เหมาะกับข้อมูลทั่วไป
- **Manhattan ($L_1$):** วัดระยะเลี้ยวตามบล็อก เหมาะกับข้อมูลมิติสูง (High-dimensional) หรือข้อมูลแบบ Grid


---
## 12. สรุปภาพรวมและสิ่งที่ได้เรียนรู้ (Summary)

```
       [ จุดข้อมูลใหม่ (Unknown) ]
                   │
                   ▼
  1. คำนวณระยะทางด้วย Pandas ((x1-x2)^2 + (y1-y2)^2)^0.5
                   │
                   ▼
  2. จัดเรียงระยะห่าง df.sort_values(by='Distance')
                   │
                   ▼
  3. ดึงเพื่อนบ้าน k ตัวแรก df.head(k)
                   │
                   ▼
  4. โหวตหาเสียงส่วนใหญ่ neighbors['Class'].mode()[0]
                   │
                   ▼
         [ ได้คำตอบของคลาส! ]
```

### ✅ Checklist สำคัญที่ต้องจำ:
1. **k-NN เป็น Lazy Learner:** ไม่มีการเทรนเพื่อหาพารามิเตอร์ล่วงหน้า แค่บันทึกข้อมูลไว้แล้วคำนวณระยะทางตอนมีคำถามเข้ามา
2. **Feature Scaling สำคัญที่สุด:** หากไม่สเกล ฟีเจอร์ที่ตัวเลขหลักร้อยจะครอบงำฟีเจอร์ตัวเลขหลักหน่วย
3. **การเลือกค่า $k$:**
   - $k=1$: เสี่ยงต่อ Overfitting
   - $k$ มากเกินไป: เสี่ยงต่อ Underfitting
   - แนะนำให้ใช้ **เลขคี่** ในปัญหา 2 คลาส เพื่อป้องกันคะแนนโหวตเสมอ
4. **ความเรียบง่าย:** อัลกอริทึมทั้งหมดสามารถเขียนได้ด้วยโค้ด Pandas ไม่กี่บรรทัดตามที่เห็นใน Lab นี้!
